In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
from typing import List

/Users/koiita/Downloads/legal_rag_project-/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
from google import genai
from dotenv import load_dotenv
import os


In [2]:
from data_loader import load_and_clean_pdf
from text_splitter import split_by_article

In [29]:
load_dotenv('../.env')
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [4]:
model=SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6978.99it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
list_of_strings = split_by_article(load_and_clean_pdf('../data/raw/158-vbhn-vpqh.pdf'))

In [17]:
vectors = model.encode(list_of_strings).tolist()
vectors

[[-0.04797658324241638,
  0.1897183656692505,
  -0.29858502745628357,
  -0.0833364725112915,
  0.1896420568227768,
  0.32266658544540405,
  -0.18807104229927063,
  0.0037677139043807983,
  -0.07745020091533661,
  0.30432409048080444,
  0.029687892645597458,
  -0.02309260703623295,
  0.1838829666376114,
  0.002063896507024765,
  0.13832537829875946,
  -0.002704195212572813,
  -0.2729502022266388,
  -0.18961673974990845,
  -0.22883298993110657,
  0.13807082176208496,
  0.05511771887540817,
  -0.06728094071149826,
  -0.16515842080116272,
  -0.09339677542448044,
  0.22179734706878662,
  -0.175227090716362,
  -0.15569272637367249,
  -0.09977853298187256,
  -0.12355481833219528,
  -0.15980404615402222,
  0.06639182567596436,
  0.05086627975106239,
  0.16512064635753632,
  0.18617971241474152,
  0.04937337338924408,
  -0.10684265941381454,
  0.17609542608261108,
  -0.1353996992111206,
  -0.0009439364075660706,
  0.05491812899708748,
  -0.11570662260055542,
  0.056411340832710266,
  -0.1494194

In [ ]:
client=chromadb.PersistentClient(path="../db/chroma_db")
collection=client.get_or_create_collection(name="test")


In [20]:
id=["luat_"+str(i) for i in range(len(list_of_strings))]

In [21]:
collection.add(
documents=list_of_strings,
embeddings=vectors,
ids=id
)

In [24]:
cau_hoi = "Bao thanh toán là gì?"
vector_cau_hoi=model.encode([cau_hoi])
ket_qua=collection.query(query_embeddings=vector_cau_hoi,n_results=3)
for i in range(3):
    print(ket_qua['ids'][0][i]+":"+ket_qua["documents"][0][i])

luat_4:Điều 4. Giải thích từ ngữ Trong Luật này, các từ ngữ dưới đây được hiểu như sau: 1. Bao thanh toán là hình thức cấp tín dụng thông qua việc mua lại khoản phải thu của bên bán hoặc ứng trước tiền thanh toán thay cho bên mua theo hợp đồng mua, bán hàng hóa, cung ứng dịch vụ giữa bên mua và bên bán. 2. Bảo lãnh ngân hàng là hình thức cấp tín dụng cho khách hàng thông qua việc tổ chức tín dụng, chi nhánh ngân hàng nước ngoài cam kết với bên nhận bảo lãnh về việc sẽ thực hiện nghĩa vụ tài chính thay cho bên có nghĩa vụ khi bên có nghĩa vụ không thực hiện hoặc thực hiện không đầy đủ nghĩa vụ đã cam kết; khách hàng phải nhận nợ bắt buộc và hoàn trả cho tổ chức tín dụng, chi nhánh ngân hàng nước ngoài theo thỏa thuận. 3. Can thiệp sớm là việc Ngân hàng Nhà nước Việt Nam (sau đây gọi là Ngân hàng Nhà nước) áp dụng các yêu cầu, biện pháp hạn chế đối với tổ chức tín dụng, chi nhánh ngân hàng nước ngoài và yêu cầu tổ chức tín dụng, chi nhánh ngân hàng

nước ngoài đó thực hiện phương án khắc

In [30]:
client=genai.Client(api_key=GEMINI_API_KEY)

In [31]:
cau_lenh = "Xin chào, bạn là ai?"
response=client.models.generate_content(model='gemini-2.5-flash',contents=cau_lenh)
print(response.text)

Xin chào! Tôi là một mô hình ngôn ngữ lớn, được huấn luyện bởi Google.


In [34]:
prompt_template = f"""
Bạn là một chuyên gia tư vấn luật ngân hàng xuất sắc. 
Nhiệm vụ của bạn là trả lời câu hỏi của người dùng một cách chính xác, ngắn gọn và dễ hiểu, CHỈ DỰA VÀO phần "Tài liệu tham khảo" được cung cấp dưới đây.
Nếu tài liệu không chứa câu trả lời, hãy nói "Tôi không tìm thấy thông tin trong tài liệu pháp lý hiện tại", tuyệt đối không tự bịa đặt thông tin.

--- TÀI LIỆU THAM KHẢO ---
{tai_lieu_tim_duoc}

--- CÂU HỎI CỦA NGƯỜI DÙNG ---
{cau_hoi}

Câu trả lời của bạn:
"""

In [33]:
tai_lieu_tim_duoc=ket_qua['documents'][0]

In [35]:
response=client.models.generate_content(model='gemini-2.5-flash',contents=prompt_template)


In [37]:
print(response.text)

Bao thanh toán là hình thức cấp tín dụng thông qua việc mua lại khoản phải thu của bên bán hoặc ứng trước tiền thanh toán thay cho bên mua theo hợp đồng mua, bán hàng hóa, cung ứng dịch vụ giữa bên mua và bên bán.
